# Lesson 06 — Joining tables

In this lesson:
- Meet the **`characters`** table (35 rows, one per Disney character)
- Combine `characters` and `movies` into one result with a `JOIN`
- Difference between `INNER JOIN` and `LEFT JOIN`
- Use table aliases to keep queries readable

Time: 30–35 minutes.

This is one of the most important lessons in the course. Real data is split across many tables for good reasons (no duplication, easier to maintain). Joining them back together is how you actually answer questions.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## Meet the `characters` table

A new table in `disney_lessons`. 35 rows, one per character:

| Column | Type | Meaning |
|--------|------|---------|
| `name` | text | Character name (e.g. "Belle") |
| `movie` | text | Which movie they're from (e.g. "Beauty and the Beast") |
| `species` | text | "human", "lion", "fish", "snowman", etc. |
| `is_villain` | true/false | Whether they're a villain |
| `voice_actor` | text | Who voiced them |

A peek:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.characters
LIMIT 5

## The setup for a join

You now have **two related tables**:

- `movies` — title, studio, year, genre, runtime, box office, RT score
- `characters` — name, movie, species, is_villain, voice_actor

The link between them is **`characters.movie` matches `movies.title`**. Every character has a movie name; if you look that movie up in the `movies` table, you find the studio, year, etc.

A `JOIN` is how you say: *"for each row in characters, also pull in the matching row from movies."*

## `INNER JOIN` — match-up rows from two tables

The shape of a join query:

```
SELECT [columns from either table]
FROM [first table] AS [alias]
INNER JOIN [second table] AS [alias]
  ON [first].[column] = [second].[column]
```

The `ON` clause says **how to match rows** between the two tables.

Here's a real one — every character with their movie's release year:

In [ ]:
%%bigquery
SELECT
  characters.name,
  characters.movie,
  movies.year,
  movies.studio
FROM disney_lessons.characters
INNER JOIN disney_lessons.movies
  ON characters.movie = movies.title
LIMIT 10

For every character, you get their movie's year and studio — drawn from the `movies` table. The `ON` clause did the matching.

## Table aliases

Typing `disney_lessons.characters.name` repeatedly is annoying. **Aliases** let you give each table a short nickname:

In [ ]:
%%bigquery
SELECT
  c.name,
  c.movie,
  m.year,
  m.studio
FROM disney_lessons.characters AS c
INNER JOIN disney_lessons.movies AS m
  ON c.movie = m.title
LIMIT 10

Same query, way cleaner. Convention is short letters: `c` for characters, `m` for movies, etc.

The `AS` is optional — `FROM disney_lessons.characters c` works too. Do whatever's most readable.

## What `INNER JOIN` does — and doesn't

`INNER JOIN` keeps **only rows where the join matched**.

If a character's movie isn't in the `movies` table, they don't appear in the result. If a movie has no characters in the `characters` table, it doesn't appear either.

Concrete example: our `characters` table has rows for "T'Challa" (from Black Panther) and "Tony Stark" (from Iron Man). Both their movies *are* in the `movies` table, so they show up in the join.

But the `movies` table has *Encanto*, *Wish*, and *Soul* — and we have no characters from those. They get **dropped** from an inner join.

## `JOIN` + `WHERE`

You can filter the joined result with `WHERE`. All Marvel characters:

In [ ]:
%%bigquery
SELECT
  c.name,
  c.movie,
  m.year
FROM disney_lessons.characters AS c
INNER JOIN disney_lessons.movies AS m
  ON c.movie = m.title
WHERE m.studio = 'Marvel Studios'
ORDER BY m.year

All villains and the year their movie came out:

In [ ]:
%%bigquery
SELECT
  c.name AS villain,
  c.movie,
  m.year
FROM disney_lessons.characters AS c
INNER JOIN disney_lessons.movies AS m
  ON c.movie = m.title
WHERE c.is_villain = TRUE
ORDER BY m.year

## `JOIN` + `GROUP BY`

Joins compose with everything else. Number of characters per studio:

In [ ]:
%%bigquery
SELECT
  m.studio,
  COUNT(*) AS character_count
FROM disney_lessons.characters AS c
INNER JOIN disney_lessons.movies AS m
  ON c.movie = m.title
GROUP BY m.studio
ORDER BY character_count DESC

## `LEFT JOIN` — keep all rows from the first table

Sometimes you want to keep *every* row of the "main" table, even if it has no match in the other one. That's `LEFT JOIN`.

Show every movie, plus its characters (if any):

In [ ]:
%%bigquery
SELECT
  m.title,
  m.year,
  c.name AS character_name
FROM disney_lessons.movies AS m
LEFT JOIN disney_lessons.characters AS c
  ON c.movie = m.title
ORDER BY m.year, c.name
LIMIT 20

For movies with no character in our table (e.g., Encanto), `character_name` shows up as `null`.

The "left" in `LEFT JOIN` refers to the table mentioned in `FROM` (the one *before* the JOIN). Everything from that table is kept; the right table's columns are filled in if there's a match, or NULL if not.

## Finding rows with no match

A common pattern: `LEFT JOIN`, then filter for `IS NULL` to find rows that *didn't* match.

Movies with **no characters** in our table:

In [ ]:
%%bigquery
SELECT
  m.title,
  m.year
FROM disney_lessons.movies AS m
LEFT JOIN disney_lessons.characters AS c
  ON c.movie = m.title
WHERE c.name IS NULL
ORDER BY m.year

You'll see Encanto, Wish, Soul, etc. — movies in our `movies` table that don't have a corresponding character entry.

## `INNER` vs. `LEFT` cheat sheet

| You want… | Use |
|-----------|-----|
| Only rows where the join matches in **both** tables | `INNER JOIN` |
| **All** rows from the first table, plus matches from the second | `LEFT JOIN` |
| The reverse (all rows from the second) | `RIGHT JOIN` (less common — usually you flip the table order and use `LEFT JOIN`) |

90% of joins you'll write are `INNER` or `LEFT`.

## Quick reminders

- Aliases (`AS m`, `AS c`) are your friend. Use them.
- `%%bigquery` still just Colab plumbing. The SQL itself is universal.
- Joins compose with `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT`.

## Exercises

### Exercise 1

Show every **villain** with the **year** their movie came out and the **studio** that made it. Use an `INNER JOIN`. Sort by year.

In [ ]:
# Your query here

### Exercise 2

How many characters does each **movie** have in our table? Show `title`, `year`, and a `character_count` column. Use `INNER JOIN` and `GROUP BY`. Sort by character count, biggest first.

In [ ]:
# Your query here

### Exercise 3

Show every **movie** alongside the **count of villains** it has in our table. Include movies with **0 villains** (use `LEFT JOIN`!). Sort by year.

In [ ]:
# Your query here

### Exercise 4 (bonus)

What movies are in our `movies` table but have **no entry at all** in `characters`? Just show the title and year. (Reuse the `LEFT JOIN` + `IS NULL` pattern.)

In [ ]:
# Your query here

---

## Solutions

⚠️ **Spoilers.**

### Solution 1

In [ ]:
%%bigquery
SELECT
  c.name AS villain,
  m.year,
  m.studio
FROM disney_lessons.characters AS c
INNER JOIN disney_lessons.movies AS m
  ON c.movie = m.title
WHERE c.is_villain = TRUE
ORDER BY m.year

### Solution 2

In [ ]:
%%bigquery
SELECT
  m.title,
  m.year,
  COUNT(*) AS character_count
FROM disney_lessons.movies AS m
INNER JOIN disney_lessons.characters AS c
  ON c.movie = m.title
GROUP BY m.title, m.year
ORDER BY character_count DESC

### Solution 3

Hint for understanding: when you `LEFT JOIN` and ask `COUNT(c.name)`, you're counting rows where the right side actually matched. Movies with no villains get a count of 0.

In [ ]:
%%bigquery
SELECT
  m.title,
  m.year,
  COUNTIF(c.is_villain = TRUE) AS villain_count
FROM disney_lessons.movies AS m
LEFT JOIN disney_lessons.characters AS c
  ON c.movie = m.title
GROUP BY m.title, m.year
ORDER BY m.year

(`COUNTIF(condition)` is a GoogleSQL convenience: counts only rows where the condition is true. You can also write `COUNT(CASE WHEN c.is_villain THEN 1 END)` — same thing. We'll see `CASE WHEN` in the next lesson.)

### Solution 4

In [ ]:
%%bigquery
SELECT
  m.title,
  m.year
FROM disney_lessons.movies AS m
LEFT JOIN disney_lessons.characters AS c
  ON c.movie = m.title
WHERE c.name IS NULL
ORDER BY m.year

## What you've learned

- ✅ `INNER JOIN` to match rows in two tables
- ✅ `LEFT JOIN` to keep all rows from the first table
- ✅ Table aliases (`AS m`, `AS c`)
- ✅ `LEFT JOIN` + `IS NULL` to find unmatched rows
- ✅ Joins compose with `WHERE`, `GROUP BY`, `ORDER BY`

## Up next

So far, all your `SELECT` columns have been raw values from the table or aggregates. Next: **conditional logic** — change what shows up in a column depending on the row's values. *"If this movie is from before 2000, label it 'classic'; otherwise 'modern'."* That's `CASE WHEN`.

Open `07_conditional.ipynb`.